# 🌿 GreenHouse Fuzzy MIMO Simulator
## Controlador Fuzzy Mamdani para Estufa Agrícola

**Dissertação de Mestrado — JucemarFC (UFSC)**

---

### Resumo
Este notebook implementa um **controlador fuzzy Mamdani MIMO** (Multiple Input, Multiple Output) completo para regulação de temperatura em estufas agrícolas com sistema de resfriamento evaporativo *pad-and-fan*.

**Entradas (4):**
| Variável | Símbolo | Universo | Descrição |
|----------|---------|----------|------------|
| Erro de temperatura | E = SP − TI | [−25, +25] °C | Diferença entre setpoint e temperatura interna |
| Derivada do erro | dE | [−8, +8] °C/tick | Taxa de variação do erro |
| Temperatura externa | TE | [−10, +55] °C | Condição climática externa |
| Velocidade do vento | VV | [0, +20] m/s | Condição de vento |

**Saídas (2):**
| Variável | Símbolo | Universo | Descrição |
|----------|---------|----------|------------|
| Potência de resfriamento | REF | [0, 100] % | Controla bomba + exaustores + janelas |
| Potência de aquecimento | HEA | [0, 100] % | Controla resistências aquecedoras |

**Regras:** 54 regras Mamdani com defuzzificação por centróide de singletons  
**Modelo físico:** 7 componentes de variação de temperatura por tick de 30 segundos  
**Ação integral (PD+I):** elimina erro em estado estacionário via aluminet e boost de atuadores

---
## 📦 ETAPA 1 — Instalação de Dependências

In [ ]:
# Instalar dependências no Google Colab
!pip install numpy matplotlib pandas scikit-fuzzy -q
print('✅ Dependências instaladas com sucesso!')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from copy import deepcopy
import warnings
warnings.filterwarnings('ignore')

# Configuração de estilo
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#16213e',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#aaa',
    'ytick.color':      '#aaa',
    'text.color':       '#e0e0e0',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'grid.alpha':       0.4,
    'font.family':      'monospace',
    'font.size':        10,
})

print('✅ Bibliotecas importadas.')
print(f'   NumPy    : {np.__version__}')
print(f'   Pandas   : {pd.__version__}')
print(f'   Matplotlib: {plt.matplotlib.__version__}')

---
## 🔺 ETAPA 2 — Funções de Pertinência (Fuzzificação)

Implementação manual das funções **trapmf** (trapezoidal) e **trimf** (triangular) seguindo exatamente a especificação do motor JavaScript original.

### Equações:

**Trapezoidal** — `trapmf(x, a, b, c, d)`:
$$
\mu(x) = \begin{cases}
0 & x \leq a \text{ ou } x \geq d \\
\frac{x-a}{b-a} & a < x < b \\
1 & b \leq x \leq c \\
\frac{d-x}{d-c} & c < x < d
\end{cases}
$$

**Triangular** — `trimf(x, a, b, c)`:
$$
\mu(x) = \begin{cases}
0 & x \leq a \text{ ou } x \geq c \\
\frac{x-a}{b-a} & a < x \leq b \\
\frac{c-x}{c-b} & b < x < c
\end{cases}
$$

In [ ]:
# ============================================================
# FUNÇÕES DE PERTINÊNCIA
# ============================================================

def trapmf(x, a, b, c, d):
    """Função de pertinência trapezoidal.
    Parâmetros: a (início subida), b (platô esq), c (platô dir), d (fim descida)
    """
    if x <= a or x >= d:
        return 0.0
    if b <= x <= c:
        return 1.0
    if a < x < b:
        return (x - a) / (b - a)
    return (d - x) / (d - c)

def trimf(x, a, b, c):
    """Função de pertinência triangular.
    Parâmetros: a (base esq), b (pico), c (base dir)
    """
    if x <= a or x >= c:
        return 0.0
    if x == b:
        return 1.0
    if x < b:
        return (x - a) / (b - a)
    return (c - x) / (c - b)

def trapmf_v(xs, a, b, c, d):
    """Versão vetorizada para plotagem."""
    return np.array([trapmf(float(x), a, b, c, d) for x in xs])

def trimf_v(xs, a, b, c):
    """Versão vetorizada para plotagem."""
    return np.array([trimf(float(x), a, b, c) for x in xs])

print('✅ Funções de pertinência definidas: trapmf, trimf')

In [ ]:
# ============================================================
# DEFINIÇÃO DAS VARIÁVEIS LINGUÍSTICAS DE ENTRADA
# ============================================================
# Seguindo exatamente os parâmetros do motor JS (js/fuzzy-engine.js)

# ── ERRO  E = SP - TI  [-25 .. +25 °C] ─────────────────────
# MN (Muito Negativo): E < -4°C  → TI MUITO acima do SP → refrigeração máxima
#  N (Negativo):  -8 < E < -0.5°C → TI acima do SP    → refrigeração moderada
#  Z (Zero):      E ≈ 0 (±1.5°C) → TI ≈ SP             → manutenção mínima
#  P (Positivo):  0.5 < E < 8°C  → TI abaixo do SP    → aquecimento moderado
# MP (Muito Positivo): E > 4°C   → TI MUITO abaixo SP  → aquecimento máximo

MF_ERRO = {
    'MN': lambda e: trapmf(e, -30, -25,  -8, -4),   # Muito Negativo
    'N':  lambda e: trimf (e,  -8,  -4,  -0.5),     # Negativo
    'Z':  lambda e: trimf (e, -1.5,  0,   1.5),     # Zero
    'P':  lambda e: trimf (e,  0.5,  4,   8),       # Positivo
    'MP': lambda e: trapmf(e,   4,   8,  25,  30),  # Muito Positivo
}

# ── DELTA-ERRO  dE  [-8 .. +8 °C/tick] ──────────────────────
# N: Erro está diminuindo (sistema está se aquecendo/se resfriando para SP)
# Z: Erro estável (sistema em equilíbrio)
# P: Erro está aumentando (afastando do SP)
MF_DE = {
    'N': lambda de: trapmf(de, -8, -4, -0.8, -0.1),
    'Z': lambda de: trimf (de, -0.5,  0,  0.5),
    'P': lambda de: trapmf(de, 0.1, 0.8,  4,   8),
}

# ── TEMPERATURA EXTERNA  TE  [-10 .. 55 °C] ─────────────────
# Fria:   TE < 12°C   — inverno, noite
# Amena:  12 < TE < 28°C — primavera/outono
# Quente: TE > 24°C   — verão, calor intenso
MF_TE = {
    'Fria':   lambda t: trapmf(t, -10,  -5,   8, 16),
    'Amena':  lambda t: trimf (t,  12,  20,  28),
    'Quente': lambda t: trapmf(t,  24,  30,  45, 55),
}

# ── VELOCIDADE DO VENTO  VV  [0 .. 20 m/s] ──────────────────
# Fraco:  VV < 6 m/s  — ventilação mínima/nula
# Medio:  4-11 m/s    — ventilação moderada
# Forte:  VV > 9 m/s  — risco de danos, fechar janelas
MF_VV = {
    'Fraco':  lambda v: trapmf(v, -1,  0,  3,  6),
    'Medio':  lambda v: trimf (v,  4,  7, 11),
    'Forte':  lambda v: trapmf(v,  9, 12, 15, 20),
}

# ── SINGLETONS DAS SAÍDAS (centróides) ──────────────────────
# Representam níveis de intensidade dos atuadores
NIV = {
    'OFF':   0,    # Desligado
    'LEVE':  20,   # Ativação leve (20% da capacidade)
    'MEDIO': 40,   # Ativação média
    'ALTO':  65,   # Ativação alta
    'MAX':   90,   # Ativação máxima
}

print('✅ Variáveis linguísticas definidas:')
print('   E  (Erro): MN, N, Z, P, MP')
print('   dE (Delta-Erro): N, Z, P')
print('   TE (Temp. Externa): Fria, Amena, Quente')
print('   VV (Vento): Fraco, Medio, Forte')
print('   Singletons REF/HEA: OFF(0), LEVE(20), MEDIO(40), ALTO(65), MAX(90)')

---
## 📊 ETAPA 3 — Visualização das Funções de Pertinência

In [ ]:
# ============================================================
# PLOTAGEM DAS FUNÇÕES DE PERTINÊNCIA
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('🌿 GreenHouse Fuzzy MIMO — Funções de Pertinência das Entradas',
             fontsize=14, fontweight='bold', color='#00ff88', y=1.01)

# ── ERRO ─────────────────────────────────────────────────────
ax = axes[0, 0]
x = np.linspace(-25, 25, 500)
cores = {'MN': '#ff3355', 'N': '#ff8800', 'Z': '#00ff88', 'P': '#00aaff', 'MP': '#00eeff'}
labels = {'MN': 'Muito Negativo (MN)', 'N': 'Negativo (N)', 'Z': 'Zero (Z)',
          'P': 'Positivo (P)', 'MP': 'Muito Positivo (MP)'}
for k, fn in MF_ERRO.items():
    y = np.array([fn(xi) for xi in x])
    ax.plot(x, y, color=cores[k], linewidth=2.5, label=labels[k])
    ax.fill_between(x, 0, y, color=cores[k], alpha=0.1)
ax.set_title('E = SP − TI (Erro de Temperatura)', color='#fff', fontweight='bold')
ax.set_xlabel('Erro [°C]'); ax.set_ylabel('Grau de Pertinência μ')
ax.legend(loc='upper right', fontsize=8)
ax.set_xlim(-25, 25); ax.set_ylim(-0.05, 1.1)
ax.axvline(0, color='#444', linestyle=':', alpha=0.5)
ax.grid(True, alpha=0.3)

# ── DELTA-ERRO ──────────────────────────────────────────────
ax = axes[0, 1]
x = np.linspace(-8, 8, 400)
cores_de = {'N': '#ff3355', 'Z': '#00ff88', 'P': '#00aaff'}
labels_de = {'N': 'Negativo (N)', 'Z': 'Zero (Z)', 'P': 'Positivo (P)'}
for k, fn in MF_DE.items():
    y = np.array([fn(xi) for xi in x])
    ax.plot(x, y, color=cores_de[k], linewidth=2.5, label=labels_de[k])
    ax.fill_between(x, 0, y, color=cores_de[k], alpha=0.1)
ax.set_title('dE = E(k) − E(k−1) (Derivada do Erro)', color='#fff', fontweight='bold')
ax.set_xlabel('dE [°C/tick]'); ax.set_ylabel('Grau de Pertinência μ')
ax.legend(loc='upper right', fontsize=8)
ax.set_xlim(-8, 8); ax.set_ylim(-0.05, 1.1)
ax.axvline(0, color='#444', linestyle=':', alpha=0.5)
ax.grid(True, alpha=0.3)

# ── TEMPERATURA EXTERNA ─────────────────────────────────────
ax = axes[1, 0]
x = np.linspace(-10, 55, 500)
cores_te = {'Fria': '#00aaff', 'Amena': '#00ff88', 'Quente': '#ff3355'}
for k, fn in MF_TE.items():
    y = np.array([fn(xi) for xi in x])
    ax.plot(x, y, color=cores_te[k], linewidth=2.5, label=k)
    ax.fill_between(x, 0, y, color=cores_te[k], alpha=0.1)
ax.set_title('TE = Temperatura Externa', color='#fff', fontweight='bold')
ax.set_xlabel('Temperatura [°C]'); ax.set_ylabel('Grau de Pertinência μ')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(-10, 55); ax.set_ylim(-0.05, 1.1)
ax.grid(True, alpha=0.3)

# ── VELOCIDADE DO VENTO ─────────────────────────────────────
ax = axes[1, 1]
x = np.linspace(0, 20, 400)
cores_vv = {'Fraco': '#00ff88', 'Medio': '#ffcc00', 'Forte': '#ff3355'}
for k, fn in MF_VV.items():
    y = np.array([fn(xi) for xi in x])
    ax.plot(x, y, color=cores_vv[k], linewidth=2.5, label=k)
    ax.fill_between(x, 0, y, color=cores_vv[k], alpha=0.1)
ax.set_title('VV = Velocidade do Vento', color='#fff', fontweight='bold')
ax.set_xlabel('Velocidade [m/s]'); ax.set_ylabel('Grau de Pertinência μ')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(0, 20); ax.set_ylim(-0.05, 1.1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mf_entradas.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('💾 Salvo: mf_entradas.png')

---
## 📋 ETAPA 4 — Base de Regras Mamdani (54 Regras)

### Formato das Regras
```
(E_key, dE_key, TE_key, VV_key, REF_saída, HEA_saída)
```
- `'*'` em VV_key significa **qualquer velocidade** (μ = 1.0)
- Força da regra: `forca = min(μE, μdE, μTE, μVV)` (t-norm produto mínimo)
- Defuzzificação: **centróide de singletons** (Sugeno-like sobre Mamdani)

In [ ]:
# ============================================================
# BASE DE REGRAS — 54 REGRAS MAMDANI
# Formato: (E_key, dE_key, TE_key, VV_key, REF, HEA)
# ============================================================
# SEMÂNTICA:
#   E=MN → TI muito acima SP → refrigeração máxima (GRUPO 1)
#   E=N  → TI pouco acima SP → refrigeração moderada (GRUPO 2)
#   E=Z  → TI ≈ SP → manutenção mínima (GRUPO 3)
#   E=P  → TI pouco abaixo SP → aquecimento moderado (GRUPO 4)
#   E=MP → TI muito abaixo SP → aquecimento máximo (GRUPO 5)

REGRAS = [
    # ══════════════════════════════════════════════════════════════════════
    # GRUPO 1 — SUPERAQUECIMENTO (E=MN: TI >> SP, erro muito negativo)
    # Justificativa: Temperatura interna muito acima do setpoint requer
    # ação máxima de resfriamento. A intensidade é modulada pela TE e VV
    # para evitar overshoot e danos por vento forte.
    # ══════════════════════════════════════════════════════════════════════
    ('MN', 'N', 'Quente', 'Fraco', NIV['MAX'],  NIV['OFF']),   # R01: calor ext+TI alta, sem vento → REF MAX
    ('MN', 'N', 'Quente', 'Medio', NIV['MAX'],  NIV['OFF']),   # R02: vento médio ajuda, manter MAX
    ('MN', 'N', 'Quente', 'Forte', NIV['ALTO'], NIV['OFF']),   # R03: vento forte → fechar janelas (-MAX → ALTO)
    ('MN', 'Z', 'Quente', 'Fraco', NIV['MAX'],  NIV['OFF']),   # R04: erro estável+TE quente → MAX
    ('MN', 'Z', 'Quente', 'Medio', NIV['MAX'],  NIV['OFF']),   # R05
    ('MN', 'P', 'Quente', 'Fraco', NIV['MAX'],  NIV['OFF']),   # R06: dE=P (esfriando), mas TI ainda >> SP
    ('MN', 'P', 'Quente', 'Medio', NIV['ALTO'], NIV['OFF']),   # R07: esfriando+vento médio → ALTO
    ('MN', 'N', 'Amena',  'Fraco', NIV['MAX'],  NIV['OFF']),   # R08: TE amena mas TI muito alta → MAX
    ('MN', 'N', 'Amena',  'Medio', NIV['ALTO'], NIV['OFF']),   # R09: vento ajuda → ALTO suficiente
    ('MN', 'Z', 'Amena',  'Fraco', NIV['MAX'],  NIV['OFF']),   # R10
    ('MN', 'Z', 'Amena',  'Medio', NIV['ALTO'], NIV['OFF']),   # R11
    ('MN', 'P', 'Amena',  'Fraco', NIV['ALTO'], NIV['OFF']),   # R12: recuperando+TE amena → ALTO
    ('MN', 'P', 'Amena',  'Medio', NIV['MEDIO'],NIV['OFF']),   # R13: boa recuperação → MEDIO
    ('MN', 'N', 'Fria',   'Fraco', NIV['ALTO'], NIV['OFF']),   # R14: TE fria ajuda → ALTO basta
    ('MN', 'Z', 'Fria',   'Fraco', NIV['MEDIO'],NIV['OFF']),   # R15: TE fria+estável → MEDIO
    ('MN', 'P', 'Fria',   'Fraco', NIV['LEVE'], NIV['OFF']),   # R16: TE fria+recuperando → LEVE

    # ══════════════════════════════════════════════════════════════════════
    # GRUPO 2 — TEMPERATURA ELEVADA (E=N: TI pouco acima do SP)
    # Justificativa: Erro moderado (E=-0.5 a -8°C). Mesmo com dE=P
    # (sistema esfriando), mantemos REF alto para superar ganho solar
    # e garantir convergência ao SP sem criação de novo equilíbrio.
    # ══════════════════════════════════════════════════════════════════════
    ('N', 'N', 'Quente', 'Fraco', NIV['ALTO'],  NIV['OFF']),   # R17: TI cai+TE quente → manter ALTO
    ('N', 'N', 'Quente', 'Medio', NIV['MEDIO'], NIV['OFF']),   # R18: vento ajuda, reduzir
    ('N', 'Z', 'Quente', 'Fraco', NIV['ALTO'],  NIV['OFF']),   # R19
    ('N', 'Z', 'Quente', 'Medio', NIV['ALTO'],  NIV['OFF']),   # R20: ALTO para superar solar
    ('N', 'P', 'Quente', 'Fraco', NIV['ALTO'],  NIV['OFF']),   # R21: esfriando mas TE quente → ALTO
    ('N', 'P', 'Quente', 'Medio', NIV['MEDIO'], NIV['OFF']),   # R22: com vento MEDIO basta
    ('N', 'N', 'Amena',  'Fraco', NIV['MEDIO'], NIV['OFF']),   # R23
    ('N', 'Z', 'Amena',  'Fraco', NIV['MEDIO'], NIV['OFF']),   # R24
    ('N', 'P', 'Amena',  'Fraco', NIV['MEDIO'], NIV['OFF']),   # R25: MEDIO para convergir
    ('N', 'N', 'Fria',   'Fraco', NIV['LEVE'],  NIV['OFF']),   # R26: TE fria → LEVE basta
    ('N', 'Z', 'Fria',   'Fraco', NIV['LEVE'],  NIV['OFF']),   # R27
    ('N', 'P', 'Fria',   'Fraco', NIV['OFF'],   NIV['OFF']),   # R28: já esfriando+TE fria → OFF

    # ══════════════════════════════════════════════════════════════════════
    # GRUPO 3 — EQUILÍBRIO (E=Z: TI ≈ SP ±1.5°C)
    # Justificativa: Sistema no setpoint. Com TE quente, manter
    # refrigeração LEVE para compensar ganho solar contínuo.
    # Com TE amena ou fria, desligar ou iniciar aquecimento.
    # '* ' = qualquer velocidade de vento (não relevante no equilíbrio)
    # ══════════════════════════════════════════════════════════════════════
    ('Z', 'N', 'Quente', '*', NIV['LEVE'],  NIV['OFF']),       # R29: esfriando+TE quente → LEVE
    ('Z', 'Z', 'Quente', '*', NIV['LEVE'],  NIV['OFF']),       # R30: estável+TE quente → LEVE refrig
    ('Z', 'P', 'Quente', '*', NIV['OFF'],   NIV['OFF']),       # R31: aquecendo leve → aguardar
    ('Z', 'N', 'Amena',  '*', NIV['OFF'],   NIV['OFF']),       # R32: esfriando+TE amena → OFF
    ('Z', 'Z', 'Amena',  '*', NIV['OFF'],   NIV['OFF']),       # R33: SETPOINT ATINGIDO — EQUILÍBRIO
    ('Z', 'P', 'Amena',  '*', NIV['OFF'],   NIV['LEVE']),      # R34: aquecendo leve+TE amena → HEA LEVE
    ('Z', 'N', 'Fria',   '*', NIV['OFF'],   NIV['OFF']),       # R35
    ('Z', 'Z', 'Fria',   '*', NIV['OFF'],   NIV['LEVE']),      # R36: estável+TE fria → HEA LEVE
    ('Z', 'P', 'Fria',   '*', NIV['OFF'],   NIV['MEDIO']),     # R37: aquecendo+TE fria → HEA MEDIO

    # ══════════════════════════════════════════════════════════════════════
    # GRUPO 4 — TEMPERATURA BAIXA (E=P: TI pouco abaixo do SP)
    # Justificativa: TI abaixo do setpoint. Aquecimento proporcional
    # à TE e dE. Com TE fria e dE=P (temperatura subindo), requer
    # mais energia. Com TE quente, pouco aquecimento necessário.
    # ══════════════════════════════════════════════════════════════════════
    ('P', 'P', 'Fria',   '*', NIV['OFF'], NIV['ALTO']),        # R38: TI baixa+fria+piorando → ALTO
    ('P', 'Z', 'Fria',   '*', NIV['OFF'], NIV['MEDIO']),       # R39: TI baixa+fria+estável → MEDIO
    ('P', 'N', 'Fria',   '*', NIV['OFF'], NIV['LEVE']),        # R40: TI baixa+fria+melhorando → LEVE
    ('P', 'P', 'Amena',  '*', NIV['OFF'], NIV['MEDIO']),       # R41: TI baixa+amena+piorando → MEDIO
    ('P', 'Z', 'Amena',  '*', NIV['OFF'], NIV['LEVE']),        # R42: TI baixa+amena+estável → LEVE
    ('P', 'N', 'Amena',  '*', NIV['OFF'], NIV['OFF']),         # R43: TI baixa+amena+melhorando → OFF
    ('P', 'P', 'Quente', '*', NIV['OFF'], NIV['LEVE']),        # R44: TI baixa mas TE quente → LEVE
    ('P', 'Z', 'Quente', '*', NIV['OFF'], NIV['OFF']),         # R45: TE quente aquece naturalmente

    # ══════════════════════════════════════════════════════════════════════
    # GRUPO 5 — SUB-RESFRIAMENTO (E=MP: TI muito abaixo do SP)
    # Justificativa: TI muito abaixo do setpoint. Risco de dano às
    # plantas por baixa temperatura. Aquecimento máximo necessário.
    # Com TE quente, a natureza já contribui; menos energia necessária.
    # ══════════════════════════════════════════════════════════════════════
    ('MP', 'P', 'Fria',   '*', NIV['OFF'], NIV['MAX']),        # R46: TI muito baixa+fria+piorando → MAX
    ('MP', 'Z', 'Fria',   '*', NIV['OFF'], NIV['MAX']),        # R47: TI muito baixa+fria+estável → MAX
    ('MP', 'N', 'Fria',   '*', NIV['OFF'], NIV['ALTO']),       # R48: melhorando → ALTO basta
    ('MP', 'P', 'Amena',  '*', NIV['OFF'], NIV['ALTO']),       # R49: muito baixa+amena → ALTO
    ('MP', 'Z', 'Amena',  '*', NIV['OFF'], NIV['MEDIO']),      # R50
    ('MP', 'N', 'Amena',  '*', NIV['OFF'], NIV['LEVE']),       # R51: melhorando rapidamente → LEVE
    ('MP', 'P', 'Quente', '*', NIV['OFF'], NIV['MEDIO']),      # R52: TE quente contribui → MEDIO
    ('MP', 'Z', 'Quente', '*', NIV['OFF'], NIV['LEVE']),       # R53: TE quente+estável → LEVE
    ('MP', 'N', 'Quente', '*', NIV['OFF'], NIV['OFF']),        # R54: TE quente já aquece naturalmente
]

print(f'✅ Base de regras carregada: {len(REGRAS)} regras Mamdani')
print(f'   Grupo 1 (E=MN, superaquecimento): R01–R16 ({sum(1 for r in REGRAS if r[0]=="MN")} regras)')
print(f'   Grupo 2 (E=N, temp. elevada):    R17–R28 ({sum(1 for r in REGRAS if r[0]=="N")} regras)')
print(f'   Grupo 3 (E=Z, equilíbrio):       R29–R37 ({sum(1 for r in REGRAS if r[0]=="Z")} regras)')
print(f'   Grupo 4 (E=P, temp. baixa):      R38–R45 ({sum(1 for r in REGRAS if r[0]=="P")} regras)')
print(f'   Grupo 5 (E=MP, sub-resfr.):      R46–R54 ({sum(1 for r in REGRAS if r[0]=="MP")} regras)')

---
## ⚙️ ETAPA 5 — Motor de Inferência Mamdani

### Pipeline de Inferência
```
Entradas (TI, TE, VV, SP)
    │
    ▼
Fuzzificação → μE, μdE, μTE, μVV
    │
    ▼
Avaliação das 54 regras (t-norm: mínimo)
    │
    ▼
Agregação (somatório ponderado de singletons)
    │
    ▼
Defuzzificação (centróide) → refCrisp, heaCrisp
    │
    ▼
Anti-windup (zona morta ±0.3°C)
    │
    ▼
Mapeamento REF/HEA → Atuadores físicos
```

In [ ]:
# ============================================================
# MOTOR DE INFERÊNCIA MAMDANI
# ============================================================

class FuzzyEngine:
    """Motor Fuzzy Mamdani MIMO para controle de temperatura de estufa."""
    
    def __init__(self):
        self.erro_anterior = None
        self.call_count = 0
    
    def fuzzificar(self, erro, dErro, TE, VV):
        """Etapa 1: Fuzzificação — calcula graus de pertinência de todas as entradas."""
        mu_E  = {k: max(0.0, min(1.0, fn(erro)))  for k, fn in MF_ERRO.items()}
        mu_dE = {k: max(0.0, min(1.0, fn(dErro))) for k, fn in MF_DE.items()}
        mu_TE = {k: max(0.0, min(1.0, fn(TE)))    for k, fn in MF_TE.items()}
        mu_VV = {k: max(0.0, min(1.0, fn(VV)))    for k, fn in MF_VV.items()}
        return mu_E, mu_dE, mu_TE, mu_VV
    
    def inferir(self, mu_E, mu_dE, mu_TE, mu_VV):
        """Etapa 2+3: Inferência Mamdani + Agregação.
        T-norm: produto mínimo (min)
        Agregação: soma ponderada de singletons (preparação para centróide)
        """
        ref_num, ref_den = 0.0, 0.0
        hea_num, hea_den = 0.0, 0.0
        regras_ativas = []
        
        for i, (eKey, deKey, teKey, vvKey, ref_centro, hea_centro) in enumerate(REGRAS):
            mu_e_val  = mu_E.get(eKey,  0.0)
            mu_de_val = mu_dE.get(deKey, 0.0)
            mu_te_val = mu_TE.get(teKey, 0.0)
            mu_vv_val = 1.0 if vvKey == '*' else mu_VV.get(vvKey, 0.0)
            
            # T-norm mínimo (Mamdani)
            forca = min(mu_e_val, mu_de_val, mu_te_val, mu_vv_val)
            
            if forca > 0.005:  # limiar de ativação
                ref_num += ref_centro * forca
                ref_den += forca
                hea_num += hea_centro * forca
                hea_den += forca
                
                tipo = ('aquecimento' if hea_centro > 0
                        else ('resfriamento' if ref_centro > 0 else 'equilibrio'))
                regras_ativas.append({
                    'id':    f'R{i+1:02d}',
                    'tipo':  tipo,
                    'forca': round(forca, 3),
                    'REF':   ref_centro,
                    'HEA':   hea_centro,
                    'desc':  f'E={eKey} dE={deKey} TE={teKey} VV={vvKey} → REF={ref_centro}% HEA={hea_centro}%',
                })
        
        return ref_num, ref_den, hea_num, hea_den, regras_ativas
    
    def defuzzificar(self, ref_num, ref_den, hea_num, hea_den, erro):
        """Etapa 4: Defuzzificação por centróide de singletons.
        
        Fórmula do centróide:
          REF* = Σ(REF_i × α_i) / Σ(α_i)
          HEA* = Σ(HEA_i × α_i) / Σ(α_i)
        onde α_i é a força da regra i.
        
        Anti-windup:
          E < -0.3°C → apenas REF (desliga HEA)
          E > +0.3°C → apenas HEA (desliga REF)
          |E| ≤ 0.3°C → zona morta (desliga ambos)
        """
        ref_crisp = ref_num / ref_den if ref_den > 0 else 0.0
        hea_crisp = hea_num / hea_den if hea_den > 0 else 0.0
        
        # Anti-windup: zona morta ±0.3°C
        if erro < -0.3:
            hea_crisp = 0.0  # TI > SP → só resfriar
        elif erro > 0.3:
            ref_crisp = 0.0  # TI < SP → só aquecer
        else:
            ref_crisp = 0.0
            hea_crisp = 0.0  # zona morta → desliga tudo
        
        return max(0.0, min(100.0, ref_crisp)), max(0.0, min(100.0, hea_crisp))
    
    def map_ref_to_atuadores(self, ref):
        """Mapeamento REF(%) → percentuais dos atuadores de resfriamento.
        
        Sequência de ativação progressiva:
          REF > 2%  → Janelas laterais (proporcionais ao REF)
          REF > 10% → Bomba + Exaustores (pad-and-fan completo)
          REF > 20% → Janelas superiores
          REF > 30% → Aluminet (bloqueia radiação solar)
        """
        s = dict(bomba=0, exaustor=0, jan_lat=0, jan_sup=0, aluminet=0)
        if ref <= 2:
            return s
        s['jan_lat']  = min(100, ref * 1.1)
        if ref > 20: s['jan_sup']  = min(100, (ref - 20) * 2.5)
        if ref > 10: s['bomba']    = min(100, (ref - 10) * 1.25)
        if ref > 10: s['exaustor'] = min(100, (ref - 10) * 1.25)
        if ref > 30: s['aluminet'] = min(100, (ref - 30) * 2.5)
        return s
    
    def map_hea_to_atuadores(self, hea):
        """Mapeamento HEA(%) → atuadores de aquecimento."""
        s = dict(aquecedor=0, jan_lat=0, jan_sup=0, aluminet=0)
        if hea <= 2:
            return s
        s['aquecedor'] = min(100, hea)
        # Janelas fechadas para reter calor
        return s
    
    def calcular_estagio_aquecedor(self, pct):
        """Converte % em número de aquecedores ativos (0–4)."""
        if pct < 5:  return {'num': 0, 'nome': 'DESLIGADO'}
        if pct < 28: return {'num': 1, 'nome': 'EST. 1'}
        if pct < 53: return {'num': 2, 'nome': 'EST. 2'}
        if pct < 78: return {'num': 3, 'nome': 'EST. 3'}
        return             {'num': 4, 'nome': 'MÁXIMO'}
    
    def computar(self, TI, TE, VV, SP):
        """Pipeline completo de inferência fuzzy.
        
        Retorna dict com: saidas, regras_ativas, mu_E, mu_dE, mu_TE, mu_VV
        """
        erro  = SP - TI
        dErro = (erro - self.erro_anterior) if self.erro_anterior is not None else 0.0
        self.erro_anterior = erro
        self.call_count += 1
        
        # 1. Fuzzificação
        mu_E, mu_dE, mu_TE, mu_VV = self.fuzzificar(erro, dErro, TE, VV)
        
        # 2+3. Inferência + Agregação
        ref_num, ref_den, hea_num, hea_den, regras_ativas = self.inferir(mu_E, mu_dE, mu_TE, mu_VV)
        
        # 4. Defuzzificação
        ref_crisp, hea_crisp = self.defuzzificar(ref_num, ref_den, hea_num, hea_den, erro)
        
        # 5. Mapeamento para atuadores
        a_ref = self.map_ref_to_atuadores(ref_crisp)
        a_hea = self.map_hea_to_atuadores(hea_crisp)
        
        saidas = {
            'bomba':     a_ref['bomba']    if ref_crisp > 0 else 0,
            'exaustor':  a_ref['exaustor'] if ref_crisp > 0 else 0,
            'aquecedor': a_hea['aquecedor'] if hea_crisp > 0 else 0,
            'jan_lat':   a_ref['jan_lat']  if ref_crisp > 0 else 0,
            'jan_sup':   a_ref['jan_sup']  if ref_crisp > 0 else 0,
            'aluminet':  a_ref['aluminet'] if ref_crisp > 0 else 0,
            'ref_crisp': ref_crisp,
            'hea_crisp': hea_crisp,
            'erro':      erro,
            'dErro':     dErro,
        }
        
        # Proteção vento forte (VV ≥ 12 m/s): fechar janelas
        if VV >= 12:
            saidas.update({'jan_lat': 0, 'jan_sup': 0, 'exaustor': 0, 'bomba': 0})
        
        return {
            'saidas': saidas,
            'regras_ativas': regras_ativas,
            'mu_E': mu_E, 'mu_dE': mu_dE, 'mu_TE': mu_TE, 'mu_VV': mu_VV,
            'erro': erro, 'dErro': dErro,
        }
    
    def reset(self):
        self.erro_anterior = None
        self.call_count = 0


# Instância global
fuzzy_engine = FuzzyEngine()

# ── Teste rápido ─────────────────────────────────────────────
r = fuzzy_engine.computar(TI=30, TE=35, VV=2, SP=25)
print('✅ Motor Fuzzy instanciado e testado.')
print(f'   TI=30°C, TE=35°C, VV=2m/s, SP=25°C:')
print(f'   Erro  = {r["erro"]:.2f}°C')
print(f'   REF*  = {r["saidas"]["ref_crisp"]:.1f}%')
print(f'   Bomba = {r["saidas"]["bomba"]:.1f}%')
print(f'   Regras ativas: {len(r["regras_ativas"])}')
fuzzy_engine.reset()

---
## 🌡️ ETAPA 6 — Modelo Físico Térmico da Estufa

### Equação Geral
$$
\Delta T_{total}(k) = \underbrace{\Delta T_{cond}}_{\text{condução}} + \underbrace{\Delta T_{solar}}_{\text{radiação}} + \underbrace{\Delta T_{vent}}_{\text{ventilação}} + \underbrace{\Delta T_{evap}}_{\text{evaporação}} + \underbrace{\Delta T_{super}}_{\text{superresfr.}} + \underbrace{\Delta T_{aq}}_{\text{aquecedor}} + \underbrace{\Delta T_{integ}}_{\text{integral}}
$$

### Temperatura Pad-and-Fan
$$
T_{BU} = T_E - (1 - HR) \cdot 12 \qquad T_{pad} = T_E - 0{,}88 \cdot (T_E - T_{BU})
$$

com $HR = 0{,}55$ (umidade relativa típica).

In [ ]:
# ============================================================
# MODELO TÉRMICO DA ESTUFA — 7 COMPONENTES
# ============================================================
# Parâmetros físicos calibrados para estufa 8×14×6m

# Parâmetros físicos
DT_SIMULACAO = 30    # segundos por tick
HR           = 0.55  # Umidade Relativa típica
EVAP_EFF     = 0.88  # Eficiência evaporativa do pad

# Constantes de tempo (segundos)
TAU_COND    = 8000   # Condução natural pelas paredes (≈133 min)
TAU_VENT_B  = 1600   # Base ventilação forçada
TAU_EVAP_B  = 280    # Base resfriamento evaporativo
TAU_SUPER_B = 35     # Base superresfriamento (ciclo de compressão)
TAU_AQ_B    = 700    # Base aquecimento elétrico
DT_AQ_MAX   = 35     # °C acima de TE que os aquecedores alcançam
GAP_NORM    = 5      # Normalização gap T_pad − SP

# Parâmetros da ação integral
INT_MAX       = 60   # Anti-windup: limite do acumulador
BOMBA_INT_MIN = 55   # % mínimo forçado da bomba (anti-offset)
AQ_BOOST_MAX  = 3.0  # Boost máx do aquecedor (TAU/3)


def calcular_T_pad(TE):
    """Temperatura do ar após o pad evaporativo.
    
    T_BU = T_E - (1 - HR) * 12   [Bulbo Úmido aproximado]
    T_pad = T_E - eff * (T_E - T_BU)
    """
    T_bwb = TE - (1 - HR) * 12
    T_pad = TE - EVAP_EFF * (TE - T_bwb)
    return T_pad


def calcular_nova_temperatura(state):
    """Calcula nova TI após um tick de DT_SIMULACAO segundos.
    
    7 componentes de variação térmica:
    1. dT_cond   — Condução natural pelas paredes (arrasta TI → TE)
    2. dT_solar  — Ganho solar (depende de aluminet)
    3. dT_vent   — Ventilação forçada (janelas + exaustores)
    4. dT_evap   — Resfriamento evaporativo (bomba + exaustor, até T_pad)
    5. dT_super  — Superresfriamento proporcional ao gap/dist (abaixo T_pad)
    6. dT_aq     — Aquecedores elétricos
    7. dT_integ  — Ação integral (boost anti-offset)
    """
    TI  = state['TI']
    TE  = state['TE']
    VV  = state['VV']
    SP  = state['SP']
    
    # ── 1. CONDUÇÃO NATURAL ────────────────────────────────────
    # dT_cond = (TE - TI) * dt / TAU_COND
    dT_cond = (TE - TI) * DT_SIMULACAO / TAU_COND
    
    # ── 2. GANHO SOLAR ─────────────────────────────────────────
    # Bloqueado progressivamente pelo aluminet (eficiência 88%)
    alm_f = state['aluminet_pct'] / 100.0
    if   TE >= 28: solar_base = 0.04
    elif TE >= 20: solar_base = 0.025
    elif TE >= 12: solar_base = 0.008
    else:          solar_base = 0.0
    dT_solar = solar_base * (1 - alm_f * 0.88)
    
    # ── 3. VENTILAÇÃO NATURAL / FORÇADA ────────────────────────
    jan_lat_f   = state['jan_lat_pct']   / 100.0
    jan_sup_f   = 1.0 if state.get('jan_sup_aberta', False) else 0.0
    exaust_f_v  = state['exaustor_pct'] / 100.0
    vf_total    = jan_lat_f * 0.6 + jan_sup_f * 0.3 + exaust_f_v * 0.5
    
    dT_vent = 0.0
    if vf_total > 0.01 and TI > TE:
        tau_v  = TAU_VENT_B / (1 + vf_total * 3.5 + min(2, VV / 4))
        dT_vent = (TE - TI) * DT_SIMULACAO / tau_v
    
    # ── 4. RESFRIAMENTO EVAPORATIVO (Pad-and-Fan) ──────────────
    T_pad   = calcular_T_pad(TE)
    bomba_f = state['bomba_pct']    / 100.0
    exaust_f = state['exaustor_pct'] / 100.0
    
    dT_evap  = 0.0
    dT_super = 0.0
    
    if bomba_f > 0.02 and exaust_f > 0.02:
        # Evaporação direta: resfria até T_pad
        if TI > T_pad:
            tau_e   = TAU_EVAP_B / (bomba_f * exaust_f)
            dT_evap = (T_pad - TI) * DT_SIMULACAO / tau_e
        
        # Superresfriamento: empurra TI → SP quando TI > SP
        # CASO A: T_pad > SP → gap positivo → força proporcional ao gap
        # CASO B: T_pad < SP → força proporcional à distância TI-SP
        if TI > SP:
            gap  = T_pad - SP
            dist = TI - SP
            
            if gap >= 0:
                # CASO A: T_pad > SP (ex: Verão TE=35 → T_pad=30.2 > SP=25)
                force = min(1.0, gap / GAP_NORM) * bomba_f * exaust_f
            else:
                # CASO B: T_pad < SP (ex: Quente_Seco TE=28 → T_pad=23.2 < SP=25)
                dist_norm = min(1.0, dist / 4.0)
                force = dist_norm * bomba_f * exaust_f * 0.6
            
            if force > 0.005:
                tau_s    = TAU_SUPER_B / force
                dT_super = -dist * DT_SIMULACAO / tau_s
    
    # ── 5. AQUECEDORES ELÉTRICOS ───────────────────────────────
    dT_aq = 0.0
    aq_n  = state.get('aquecedor_num', 0)
    if aq_n > 0:
        T_tgt = min(TE + DT_AQ_MAX, 60)
        tau_a = TAU_AQ_B / (aq_n / 4.0)
        dT_aq = (T_tgt - TI) * DT_SIMULACAO / tau_a
    
    # ── 6. AÇÃO INTEGRAL (anti-offset) ─────────────────────────
    E_atual   = SP - TI
    dT_integ  = 0.0
    
    if E_atual < -0.3 and E_atual > -2.5:
        # Integrador de resfriamento (TI acima do SP)
        state['integral_ref'] = min(INT_MAX, state['integral_ref'] + abs(E_atual))
        state['integral_hea'] = 0.0
        
        if state['bomba_pct'] > 2 or state['exaustor_pct'] > 2:
            frac = min(1.0, state['integral_ref'] / INT_MAX)
            bomba_min  = BOMBA_INT_MIN * frac / 100.0
            exaust_min = BOMBA_INT_MIN * frac / 100.0
            bomba_eff  = max(bomba_f,  bomba_min)
            exaust_eff = max(exaust_f, exaust_min)
            
            dT_evap_int  = 0.0
            dT_super_int = 0.0
            dist_int = TI - SP
            
            if TI > T_pad:
                tau_e_int   = TAU_EVAP_B / (bomba_eff * exaust_eff)
                dT_evap_int = (T_pad - TI) * DT_SIMULACAO / tau_e_int
            if dist_int > 0:
                gap_int   = T_pad - SP
                if gap_int >= 0:
                    force_int = min(1.0, gap_int / GAP_NORM) * bomba_eff * exaust_eff
                else:
                    force_int = min(1.0, dist_int / 4.0) * bomba_eff * exaust_eff * 0.6
                if force_int > 0.005:
                    dT_super_int = -dist_int * DT_SIMULACAO / (TAU_SUPER_B / force_int)
            
            dT_ref_boost = dT_evap_int + dT_super_int
            dT_ref_orig  = dT_evap + dT_super
            if dT_ref_boost < dT_ref_orig:
                dT_integ = dT_ref_boost - dT_ref_orig
    
    elif E_atual > 0.3 and E_atual < 2.5:
        # Integrador de aquecimento (TI abaixo do SP)
        state['integral_hea'] = min(INT_MAX, state['integral_hea'] + E_atual)
        state['integral_ref'] = 0.0
        
        if aq_n > 0:
            frac        = min(1.0, state['integral_hea'] / INT_MAX)
            boost_hea   = 1.0 + (AQ_BOOST_MAX - 1) * frac
            tau_a_eff   = (TAU_AQ_B / (aq_n / 4.0)) / boost_hea
            T_tgt_int   = min(TE + DT_AQ_MAX, 60)
            dT_aq_boost = (T_tgt_int - TI) * DT_SIMULACAO / tau_a_eff
            if dT_aq_boost > dT_aq:
                dT_integ = dT_aq_boost - dT_aq
    
    elif abs(E_atual) <= 0.3:
        # Atingiu SP: decaimento suave do integrador
        state['integral_ref'] *= 0.75
        state['integral_hea'] *= 0.75
    
    # ── 7. VARIAÇÃO TOTAL ─────────────────────────────────────
    dT_total = dT_cond + dT_solar + dT_vent + dT_evap + dT_super + dT_aq + dT_integ
    dT_sat   = max(-8.0, min(8.0, dT_total))  # saturação de segurança
    nova_TI  = max(-5.0, min(65.0, TI + dT_sat))
    
    # Retorna componentes para análise
    return nova_TI, {
        'dT_cond':  dT_cond,  'dT_solar': dT_solar,
        'dT_vent':  dT_vent,  'dT_evap':  dT_evap,
        'dT_super': dT_super, 'dT_aq':    dT_aq,
        'dT_integ': dT_integ, 'dT_total': dT_sat,
        'T_pad':    T_pad,
    }


print('✅ Modelo térmico definido (7 componentes):')
print('   1. dT_cond   — Condução natural (TAU=8000s)')
print('   2. dT_solar  — Radiação solar (aluminet bloqueia 88%)')
print('   3. dT_vent   — Ventilação forçada (janelas + exaustores)')
print('   4. dT_evap   — Evaporação pad-and-fan (até T_pad)')
print('   5. dT_super  — Superresfriamento (CASO A e B)')
print('   6. dT_aq     — Aquecedores elétricos (0–4 ativos)')
print('   7. dT_integ  — Ação integral anti-offset')

# Teste T_pad
print(f'\n   T_pad (TE=35°C, HR=55%): {calcular_T_pad(35):.1f}°C')
print(f'   T_pad (TE=28°C, HR=55%): {calcular_T_pad(28):.1f}°C')

---
## 🔄 ETAPA 7 — Loop de Simulação

### Aluminet Integral (Ação Integral sobre Sombra)
O aluminet fecha progressivamente quando o sistema está preso entre 0.3°C e 2.5°C acima do SP:
$$
\alpha_{alm}^{int} = \frac{I_{ref}}{I_{max}} \times 100\%
$$
$$
\alpha_{alm} = \max(\alpha_{alm}^{fuzzy},\ \alpha_{alm}^{int})
$$

In [ ]:
# ============================================================
# LOOP PRINCIPAL DE SIMULAÇÃO
# ============================================================

def criar_estado(TI, TE, VV, SP):
    """Cria estado inicial da simulação."""
    return {
        'TI': float(TI), 'TE': float(TE),
        'VV': float(VV), 'SP': float(SP),
        'bomba_pct': 0.0, 'exaustor_pct': 0.0,
        'aquecedor_pct': 0.0, 'jan_lat_pct': 0.0,
        'aluminet_pct': 0.0, 'jan_sup_pct': 0.0,
        'jan_sup_aberta': False, 'aquecedor_num': 0,
        'integral_ref': 0.0, 'integral_hea': 0.0,
        'tick': 0,
    }


def aplicar_aluminet_integral(state, saidas, INT_MX=60):
    """Sobrescreve aluminet do fuzzy com ação integral quando necessário.
    
    Mecanismo:
      Quando TI está presa 0.3–2.5°C ACIMA do SP:
        aluminet fecha progressivamente (0 → 100%) conforme integral_ref cresce.
      Quando TI está presa 0.3–2.5°C ABAIXO do SP:
        aluminet abre para capturar calor solar.
    """
    E_int  = state['SP'] - state['TI']
    I_ref  = state['integral_ref']
    I_hea  = state['integral_hea']
    alm    = saidas['aluminet']
    
    if E_int < -0.3 and E_int > -2.5 and I_ref > 0:
        frac   = min(1.0, I_ref / INT_MX)
        alm_int = frac * 100
        alm    = max(alm, alm_int)
    elif E_int > 0.3 and E_int < 2.5 and I_hea > 0:
        frac   = min(1.0, I_hea / INT_MX)
        alm    = max(0, alm - frac * alm)
    
    return alm


def calcular_estagio_aquecedor(pct):
    if pct < 5:  return 0
    if pct < 28: return 1
    if pct < 53: return 2
    if pct < 78: return 3
    return 4


def executar_simulacao(TI_init, TE, VV, SP, n_ticks=100,
                       nome='Cenário', verbose=False):
    """Executa loop completo de simulação Fuzzy MIMO.
    
    Parâmetros:
      TI_init   : temperatura interna inicial (°C)
      TE        : temperatura externa (°C, constante)
      VV        : velocidade do vento (m/s, constante)
      SP        : setpoint desejado (°C)
      n_ticks   : número de ticks a simular (1 tick = 30s)
      nome      : identificação do cenário
    
    Retorna: dict com históricos e métricas
    """
    state   = criar_estado(TI_init, TE, VV, SP)
    engine  = FuzzyEngine()
    
    # Históricos
    hist = {
        'tick': [], 'tempo_s': [],
        'TI': [], 'SP': [], 'TE': [],
        'bomba': [], 'exaustor': [],
        'aquecedor': [], 'jan_lat': [],
        'aluminet': [], 'ref_crisp': [],
        'hea_crisp': [], 'erro': [],
        'integral_ref': [], 'integral_hea': [],
        'dT_cond': [], 'dT_solar': [], 'dT_vent': [],
        'dT_evap': [], 'dT_super': [], 'dT_aq': [],
        'dT_integ': [], 'T_pad': [],
        'regras_n': [],
    }
    
    tick_sp     = None   # tick em que atingiu SP
    overshoot   = 0.0
    cooling_mode = (TI_init > SP)  # True = resfriando, False = aquecendo
    
    for t in range(n_ticks):
        state['tick'] = t
        
        # 1. Motor Fuzzy
        resultado = engine.computar(state['TI'], state['TE'], state['VV'], state['SP'])
        saidas    = resultado['saidas']
        
        # 2. Aplicar saídas ao estado
        state['bomba_pct']     = saidas['bomba']
        state['exaustor_pct']  = saidas['exaustor']
        state['aquecedor_pct'] = saidas['aquecedor']
        state['jan_lat_pct']   = saidas['jan_lat']
        state['jan_sup_pct']   = saidas['jan_sup']
        state['jan_sup_aberta'] = saidas['jan_sup'] > 40
        state['aquecedor_num']  = calcular_estagio_aquecedor(saidas['aquecedor'])
        
        # 3. Aluminet integral
        state['aluminet_pct'] = aplicar_aluminet_integral(state, saidas)
        
        # 4. Modelo físico
        nova_TI, comps = calcular_nova_temperatura(state)
        
        # 5. Métricas
        erro_abs = abs(SP - nova_TI)
        if tick_sp is None and erro_abs <= 0.5:
            tick_sp = t
        
        if cooling_mode:
            over = max(0, (SP - 0.5) - nova_TI)
        else:
            over = max(0, nova_TI - (SP + 0.5))
        overshoot = max(overshoot, over)
        
        # 6. Gravar histórico
        for k, v in [
            ('tick', t), ('tempo_s', t * DT_SIMULACAO),
            ('TI', nova_TI), ('SP', SP), ('TE', TE),
            ('bomba', saidas['bomba']), ('exaustor', saidas['exaustor']),
            ('aquecedor', saidas['aquecedor']), ('jan_lat', saidas['jan_lat']),
            ('aluminet', state['aluminet_pct']),
            ('ref_crisp', saidas['ref_crisp']),
            ('hea_crisp', saidas['hea_crisp']),
            ('erro', SP - nova_TI),
            ('integral_ref', state['integral_ref']),
            ('integral_hea', state['integral_hea']),
            ('regras_n', len(resultado['regras_ativas'])),
        ]:
            hist[k].append(v)
        
        for k, v in comps.items():
            if k in hist:
                hist[k].append(v)
        
        state['TI'] = nova_TI
        
        if verbose and t % 5 == 0:
            print(f'  T{t:03d} TI={nova_TI:.2f}°C  E={SP-nova_TI:+.2f}°C  '
                  f'REF={saidas["ref_crisp"]:.0f}%  HEA={saidas["hea_crisp"]:.0f}%  '
                  f'Bomba={saidas["bomba"]:.0f}%')
    
    # Métricas finais
    erros_abs = [abs(SP - ti) for ti in hist['TI']]
    mae  = np.mean(erros_abs)
    mse  = np.mean([e**2 for e in erros_abs])
    rmse = np.sqrt(mse)
    
    return {
        'nome': nome,
        'hist': hist,
        'metricas': {
            'TI_final':     hist['TI'][-1],
            'tick_sp':      tick_sp,
            'tempo_sp_s':   tick_sp * DT_SIMULACAO if tick_sp is not None else None,
            'overshoot':    overshoot,
            'MAE':          mae,
            'MSE':          mse,
            'RMSE':         rmse,
            'convergiu':    tick_sp is not None,
        },
        'params': {'TI_init': TI_init, 'TE': TE, 'VV': VV, 'SP': SP},
    }

print('✅ Loop de simulação definido.')

---
## 🧪 ETAPA 8 — Simulação de 8 Cenários de Validação

In [ ]:
# ============================================================
# DEFINIÇÃO DOS 8 CENÁRIOS
# ============================================================
CENARIOS = [
    #  nome             TI    TE   VV   SP  ticks
    ('VERÃO_INTENSO',   38,   35,  2,   25, 100),
    ('INVERNO_NOITE',    8,    3,  5,   18,  60),
    ('QUENTE_SECO',     34,   28,  3,   25, 100),
    ('NOITE_QUENTE',    30,   25,  3,   26,  50),
    ('MANHÃ_PRIMAVERA', 22,   18,  4,   22,  20),
    ('TEMPESTADE',      20,   15, 13,   20,  20),
    ('NUBLADO_FRIO',    12,    8,  6,   20,  60),
    ('CALOR_VENTO',     32,   28, 10,   25, 100),
]

print('🚀 Iniciando simulação dos 8 cenários de validação...')
print('─' * 70)

resultados = {}
for nome, TI, TE, VV, SP, ticks in CENARIOS:
    r = executar_simulacao(TI, TE, VV, SP, n_ticks=ticks, nome=nome)
    resultados[nome] = r
    m = r['metricas']
    status  = '✅' if m['convergiu'] else '❌'
    tick_s  = f'T{m["tick_sp"]:03d}' if m['convergiu'] else 'N/A'
    tempo_s = f'{m["tempo_sp_s"]}s' if m['convergiu'] else '---'
    print(f'{status} {nome:<20}  TI_init={TI:5.1f}°C  SP={SP:.0f}°C  '
          f'TI_final={m["TI_final"]:6.2f}°C  '
          f'Tick={tick_s}  MAE={m["MAE"]:5.3f}°C  Overshoot={m["overshoot"]:.2f}°C')

print('─' * 70)
print('\n🏁 Simulação concluída!')

---
## 📈 ETAPA 9 — Relatório de Visualizações

In [ ]:
# ============================================================
# FIGURA 1: TEMPERATURA × TEMPO — TODOS OS CENÁRIOS
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

cores_cenario = {
    'VERÃO_INTENSO':   '#ff3355',
    'INVERNO_NOITE':   '#00aaff',
    'QUENTE_SECO':     '#ff8800',
    'NOITE_QUENTE':    '#ffcc00',
    'MANHÃ_PRIMAVERA': '#00ff88',
    'TEMPESTADE':      '#aa88ff',
    'NUBLADO_FRIO':    '#88ccff',
    'CALOR_VENTO':     '#ff6688',
}

for ax, (nome, _, _, _, SP, _) in zip(axes, CENARIOS):
    r    = resultados[nome]
    hist = r['hist']
    m    = r['metricas']
    t_min = [s / 60 for s in hist['tempo_s']]
    cor  = cores_cenario[nome]
    
    ax.plot(t_min, hist['TI'], color=cor,    linewidth=2.5, label='TI (medida)')
    ax.plot(t_min, hist['SP'], color='#fff', linewidth=1.5, linestyle='--', label='SP (alvo)', alpha=0.7)
    ax.axhline(SP + 0.5, color='#444', linestyle=':', linewidth=1)
    ax.axhline(SP - 0.5, color='#444', linestyle=':', linewidth=1)
    
    if m['tick_sp'] is not None:
        ax.axvline(m['tick_sp'] * DT_SIMULACAO / 60, color='#00ff88',
                   linestyle='-.', linewidth=1.5, alpha=0.8, label=f'SP atingido')
    
    status  = '✅' if m['convergiu'] else '❌'
    tick_sp_lbl = f'T{m["tick_sp"]}' if m['convergiu'] else 'N/A'
    ax.set_title(f'{status} {nome}\nTfinal={m["TI_final"]:.2f}°C  {tick_sp_lbl}',
                 fontsize=9, fontweight='bold', color='#e0e0e0')
    ax.set_xlabel('Tempo [min]', fontsize=8)
    ax.set_ylabel('Temperatura [°C]', fontsize=8)
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.3)

fig.suptitle('🌿 GreenHouse Fuzzy MIMO — Resposta de Temperatura (8 Cenários)',
             fontsize=13, fontweight='bold', color='#00ff88')
plt.tight_layout()
plt.savefig('resposta_temperatura.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('💾 Salvo: resposta_temperatura.png')

In [ ]:
# ============================================================
# FIGURA 2: ATUADORES — CENÁRIO VERÃO (MAIS CRÍTICO)
# ============================================================
nome_sel = 'VERÃO_INTENSO'
r    = resultados[nome_sel]
hist = r['hist']
t_min = [s / 60 for s in hist['tempo_s']]

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle(f'🌿 GreenHouse Fuzzy MIMO — Atuadores: {nome_sel}\n'
             f'(TI={r["params"]["TI_init"]}°C → SP={r["params"]["SP"]}°C, TE={r["params"]["TE"]}°C)',
             fontsize=12, fontweight='bold', color='#ff3355')

plots = [
    ('TI / SP',         t_min, [hist['TI'], hist['SP']],     ['TI medida', 'Setpoint'],  ['#ff3355', '#fff']),
    ('REF crisp (%)',   t_min, [hist['ref_crisp']],          ['REF fuzzy'],               ['#00aaff']),
    ('Bomba (%)',       t_min, [hist['bomba']],               ['Bomba d\'água'],          ['#00eeff']),
    ('Exaustor (%)',    t_min, [hist['exaustor']],            ['Exaustor'],                ['#ffcc00']),
    ('Aluminet (%)',    t_min, [hist['aluminet']],            ['Aluminet (sombra)'],       ['#aa88ff']),
    ('Integral Ref',   t_min, [hist['integral_ref']],        ['Integrador REF'],          ['#ff8800']),
]

for ax, (titulo, tx, ys, labels, cores) in zip(axes.flatten(), plots):
    for y, lbl, cor in zip(ys, labels, cores):
        ax.plot(tx, y, color=cor, linewidth=2, label=lbl)
    ax.set_title(titulo, fontweight='bold', color='#e0e0e0', fontsize=10)
    ax.set_xlabel('Tempo [min]', fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('atuadores_verao.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('💾 Salvo: atuadores_verao.png')

In [ ]:
# ============================================================
# FIGURA 3: COMPONENTES TÉRMICOS — ANÁLISE DETALHADA
# ============================================================
nome_sel = 'VERÃO_INTENSO'
r    = resultados[nome_sel]
hist = r['hist']
t_min = [s / 60 for s in hist['tempo_s']]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'🌡️ Componentes Térmicos — {nome_sel}', fontsize=12,
             fontweight='bold', color='#ff3355')

# Stacked area dos componentes
components = [
    ('dT_cond',  '#00aaff', 'Condução'),
    ('dT_solar', '#ffcc00', 'Solar'),
    ('dT_vent',  '#00eeff', 'Ventilação'),
    ('dT_evap',  '#00ff88', 'Evaporação'),
    ('dT_super', '#88ff88', 'Superresfr.'),
    ('dT_aq',    '#ff8800', 'Aquecedor'),
    ('dT_integ', '#ff3355', 'Integral'),
]

for key, cor, label in components:
    if key in hist and len(hist[key]) == len(t_min):
        ax1.plot(t_min, hist[key], color=cor, linewidth=1.8, label=label, alpha=0.85)

ax1.axhline(0, color='#666', linestyle='--', linewidth=1)
ax1.set_title('Componentes dT por Tick', fontweight='bold', color='#e0e0e0')
ax1.set_xlabel('Tempo [min]'); ax1.set_ylabel('ΔT [°C/tick]')
ax1.legend(fontsize=8, ncol=2); ax1.grid(True, alpha=0.3)

# Temperatura + T_pad
ax2.plot(t_min, hist['TI'],    color='#ff3355', linewidth=2.5, label='TI (interna)')
ax2.plot(t_min, hist['SP'],    color='#fff',    linewidth=1.5, linestyle='--', label='SP')
if 'T_pad' in hist and len(hist['T_pad']) == len(t_min):
    ax2.plot(t_min, hist['T_pad'], color='#00eeff', linewidth=1.5,
             linestyle=':', label=f'T_pad (evap)')
ax2.set_title('Temperatura Interna + T_pad + SP', fontweight='bold', color='#e0e0e0')
ax2.set_xlabel('Tempo [min]'); ax2.set_ylabel('Temperatura [°C]')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('componentes_termicos.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('💾 Salvo: componentes_termicos.png')

In [ ]:
# ============================================================
# FIGURA 4: ATIVAÇÃO DE REGRAS — ANÁLISE POR TICK
# ============================================================
# Simulação curta para visualizar ativação das MFs

nome_sel = 'VERÃO_INTENSO'
r    = resultados[nome_sel]
hist = r['hist']
t_min = [s / 60 for s in hist['tempo_s']]
SP   = r['params']['SP']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'🔬 Ativação das Funções de Pertinência — {nome_sel}',
             fontsize=12, fontweight='bold', color='#00eeff')

# μ(E) ao longo do tempo
ax = axes[0, 0]
erros_hist = hist['erro']
for k, cor in [('MN', '#ff3355'), ('N', '#ff8800'), ('Z', '#00ff88'),
               ('P', '#00aaff'), ('MP', '#00eeff')]:
    mu = [MF_ERRO[k](e) for e in erros_hist]
    ax.plot(t_min, mu, color=cor, linewidth=1.8, label=f'μ_E({k})', alpha=0.85)
ax.set_title('μ do Erro E = SP − TI', fontweight='bold', color='#e0e0e0')
ax.set_xlabel('Tempo [min]'); ax.set_ylabel('Grau μ')
ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)

# Quantidade de regras ativas
ax = axes[0, 1]
ax.plot(t_min, hist['regras_n'], color='#ffcc00', linewidth=2)
ax.fill_between(t_min, 0, hist['regras_n'], color='#ffcc00', alpha=0.15)
ax.set_title('Número de Regras Ativas por Tick', fontweight='bold', color='#e0e0e0')
ax.set_xlabel('Tempo [min]'); ax.set_ylabel('Qtd. Regras')
ax.grid(True, alpha=0.3)

# Superfície fuzzy (E vs dE → REF)
ax = axes[0, 2]
e_range  = np.linspace(-10, 2, 50)
de_range = np.linspace(-3, 3, 50)
E_g, DE_g = np.meshgrid(e_range, de_range)

surf = np.zeros_like(E_g)
eng_surf = FuzzyEngine()
for i in range(len(de_range)):
    for j in range(len(e_range)):
        r_surf = eng_surf.computar(SP - e_range[j], 35, 2, SP)
        surf[i, j] = r_surf['saidas']['ref_crisp']
        eng_surf.reset()

im = ax.contourf(E_g, DE_g, surf, 20, cmap='RdYlGn_r')
plt.colorbar(im, ax=ax, label='REF (%)')
ax.set_title('Superfície Fuzzy: E × dE → REF\n(TE=35°C fixo)', fontweight='bold', color='#e0e0e0')
ax.set_xlabel('Erro E [°C]'); ax.set_ylabel('dE [°C/tick]')

# Erro ao longo do tempo
ax = axes[1, 0]
ax.plot(t_min, erros_hist, color='#ff8800', linewidth=2)
ax.axhline(0.5,  color='#00ff88', linestyle='--', linewidth=1, alpha=0.7, label='±0.5°C')
ax.axhline(-0.5, color='#00ff88', linestyle='--', linewidth=1, alpha=0.7)
ax.fill_between(t_min, -0.5, 0.5, color='#00ff88', alpha=0.05)
ax.set_title('Erro E(t) = SP − TI', fontweight='bold', color='#e0e0e0')
ax.set_xlabel('Tempo [min]'); ax.set_ylabel('Erro [°C]')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# REF e HEA crips
ax = axes[1, 1]
ax.plot(t_min, hist['ref_crisp'], color='#00aaff', linewidth=2, label='REF crisp (%)')
ax.plot(t_min, hist['hea_crisp'], color='#ff8800', linewidth=2, label='HEA crisp (%)')
ax.set_title('Saídas Defuzzificadas REF e HEA', fontweight='bold', color='#e0e0e0')
ax.set_xlabel('Tempo [min]'); ax.set_ylabel('%')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Integradores
ax = axes[1, 2]
ax.plot(t_min, hist['integral_ref'], color='#ff3355', linewidth=2, label='Integral REF')
ax.plot(t_min, hist['integral_hea'], color='#00aaff', linewidth=2, label='Integral HEA')
ax.axhline(INT_MAX, color='#666', linestyle=':', linewidth=1, label=f'INT_MAX={INT_MAX}')
ax.set_title('Acumuladores Integrais (Anti-offset)', fontweight='bold', color='#e0e0e0')
ax.set_xlabel('Tempo [min]'); ax.set_ylabel('Acumulador')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ativacao_mf.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('💾 Salvo: ativacao_mf.png')

---
## 📊 ETAPA 10 — Métricas e Tabela de Convergência

In [ ]:
# ============================================================
# TABELA DE CONVERGÊNCIA — TODOS OS CENÁRIOS
# ============================================================

rows = []
for nome, TI, TE, VV, SP, ticks in CENARIOS:
    r = resultados[nome]
    m = r['metricas']
    tempo_str = f'{m["tempo_sp_s"]}s' if m['convergiu'] else '---'
    tick_str  = f'T{m["tick_sp"]:03d}' if m['convergiu'] else '---'
    T_pad_val = calcular_T_pad(TE)
    rows.append({
        'Cenário':       nome,
        'TI₀ (°C)':     TI,
        'TE (°C)':       TE,
        'VV (m/s)':      VV,
        'SP (°C)':       SP,
        'T_pad (°C)':    round(T_pad_val, 1),
        'TI_final (°C)': round(m['TI_final'], 2),
        'Tick SP':       tick_str,
        'Tempo SP':      tempo_str,
        'Overshoot (°C)':round(m['overshoot'], 2),
        'MAE (°C)':      round(m['MAE'],  3),
        'RMSE (°C)':     round(m['RMSE'], 3),
        'Convergiu':     '✅' if m['convergiu'] else '❌',
    })

df = pd.DataFrame(rows)
print('\n📊 TABELA DE CONVERGÊNCIA — GreenHouse Fuzzy MIMO v4.0')
print('=' * 120)
print(df.to_string(index=False))
print('=' * 120)
print(f'\n✅ Cenários convergidos: {sum(1 for r in rows if r["Convergiu"]=="✅")}/{len(rows)}')
print(f'📉 MAE médio global: {df["MAE (°C)"].mean():.4f}°C')
print(f'📉 RMSE médio global: {df["RMSE (°C)"].mean():.4f}°C')
print(f'📈 Overshoot máximo: {df["Overshoot (°C)"].max():.2f}°C')

# Salvar CSV
df.to_csv('convergencia_fuzzy.csv', index=False)
print('\n💾 Salvo: convergencia_fuzzy.csv')

In [ ]:
# ============================================================
# FIGURA 5: COMPARATIVO DE MÉTRICAS
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('📊 Métricas de Desempenho — GreenHouse Fuzzy MIMO',
             fontsize=13, fontweight='bold', color='#00ff88')

nomes_curtos = [n.replace('_', '\n') for n in df['Cenário']]

# MAE por cenário
ax = axes[0]
bars = ax.bar(nomes_curtos, df['MAE (°C)'], color='#00aaff', alpha=0.8, edgecolor='#fff', linewidth=0.5)
ax.axhline(0.5, color='#00ff88', linestyle='--', linewidth=1.5, label='Limiar SP (±0.5°C)')
ax.set_title('MAE por Cenário', fontweight='bold', color='#e0e0e0')
ax.set_ylabel('MAE [°C]'); ax.tick_params(axis='x', labelsize=7)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, df['MAE (°C)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=7, color='#ccc')

# Tempo de convergência
ax = axes[1]
tempos = [m['tempo_sp_s'] if m['tempo_sp_s'] is not None else 0
          for _, m in [(n, resultados[n]['metricas']) for n in df['Cenário']]]
cores_conv = ['#00ff88' if t > 0 else '#ff3355' for t in tempos]
bars = ax.bar(nomes_curtos, [t/60 for t in tempos], color=cores_conv, alpha=0.8,
              edgecolor='#fff', linewidth=0.5)
ax.set_title('Tempo de Convergência', fontweight='bold', color='#e0e0e0')
ax.set_ylabel('Tempo [min]'); ax.tick_params(axis='x', labelsize=7)
ax.grid(True, alpha=0.3, axis='y')
for bar, t in zip(bars, tempos):
    if t > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{t}s', ha='center', va='bottom', fontsize=7, color='#ccc')

# RMSE
ax = axes[2]
bars = ax.bar(nomes_curtos, df['RMSE (°C)'], color='#ff8800', alpha=0.8,
              edgecolor='#fff', linewidth=0.5)
ax.set_title('RMSE por Cenário', fontweight='bold', color='#e0e0e0')
ax.set_ylabel('RMSE [°C]'); ax.tick_params(axis='x', labelsize=7)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, df['RMSE (°C)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=7, color='#ccc')

plt.tight_layout()
plt.savefig('metricas_comparativo.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('💾 Salvo: metricas_comparativo.png')

---
## ⚖️ ETAPA 11 — Comparação Fuzzy vs Controle ON/OFF

In [ ]:
# ============================================================
# SIMULADOR ON/OFF (histerese ±2°C)
# ============================================================

def executar_onoff(TI_init, TE, VV, SP, n_ticks=100, nome='ON/OFF'):
    """Simulação de controle ON/OFF com histerese ±2°C.
    Usa o mesmo modelo térmico para comparação justa.
    """
    TI      = float(TI_init)
    T_pad   = calcular_T_pad(TE)
    hist_TI = []
    hist_t  = []
    tick_sp = None
    overshoot = 0.0
    cooling_mode = (TI_init > SP)
    energia = 0.0
    agua    = 0.0

    # Potências nominais (W)
    P_bomba     = 370
    P_exaustor  = 250
    P_aquecedor = 1500

    for t in range(n_ticks):
        # Lógica ON/OFF com histerese ±2°C
        aq_on   = TI < SP - 2.0
        cool_on = TI > SP + 2.0
        bf      = 1.0 if cool_on else 0.0
        ef      = 1.0 if cool_on else 0.0

        # Mesmo modelo térmico (simplificado)
        dT_cond = (TE - TI) * DT_SIMULACAO / TAU_COND
        if   TE >= 28: dT_solar = 0.04
        elif TE >= 20: dT_solar = 0.025
        elif TE >= 12: dT_solar = 0.008
        else:          dT_solar = 0.0

        dT_vent = 0.0
        if cool_on and TI > TE:
            vf    = 0.6 + 0.5
            tau_v = TAU_VENT_B / (1 + vf * 3.5 + min(2, VV / 4))
            dT_vent = (TE - TI) * DT_SIMULACAO / tau_v

        dT_evap = dT_super = 0.0
        if bf > 0 and ef > 0:
            if TI > T_pad:
                dT_evap = (T_pad - TI) * DT_SIMULACAO / TAU_EVAP_B
            if TI > SP:
                gap   = max(0, T_pad - SP)
                force = min(1, gap / 10) * bf * ef
                if force > 0.01:
                    dT_super = -(TI - SP) * DT_SIMULACAO / (55 / force)

        dT_aq = 0.0
        if aq_on:
            T_tgt = min(TE + DT_AQ_MAX, 60)
            dT_aq = (T_tgt - TI) * DT_SIMULACAO / TAU_AQ_B

        TI = max(-5, min(65, TI + max(-8, min(8,
            dT_cond + dT_solar + dT_vent + dT_evap + dT_super + dT_aq))))

        hist_TI.append(TI)
        hist_t.append(t * DT_SIMULACAO)

        if tick_sp is None and abs(SP - TI) <= 0.5:
            tick_sp = t

        if cooling_mode:
            over = max(0, (SP - 0.5) - TI)
        else:
            over = max(0, TI - (SP + 0.5))
        overshoot = max(overshoot, over)

        # Consumo (watts)
        P_tot = (4 * P_aquecedor if aq_on else 0) + \
                (P_bomba if cool_on else 0) + \
                (2 * P_exaustor if cool_on else 0)
        energia += P_tot * DT_SIMULACAO / 3_600_000  # kWh
        agua    += (100 * 2.5 * DT_SIMULACAO / 3600) if cool_on else 0

    erros = [abs(SP - ti) for ti in hist_TI]
    mae   = np.mean(erros)
    return {
        'TI': hist_TI, 'tempo_s': hist_t, 'TI_final': hist_TI[-1],
        'tick_sp': tick_sp,
        'tempo_sp_s': tick_sp * DT_SIMULACAO if tick_sp is not None else None,
        'overshoot': overshoot, 'MAE': mae,
        'energia_kwh': energia, 'agua_litros': agua,
        'convergiu': tick_sp is not None,
    }


# ── Rodar comparação para todos os cenários ──────────────────
print('⚖️  Comparação Fuzzy vs ON/OFF\n' + '─' * 80)
cmp_rows = []

for nome, TI, TE, VV, SP, ticks in CENARIOS:
    rf  = resultados[nome]
    ro  = executar_onoff(TI, TE, VV, SP, n_ticks=ticks)
    mf  = rf['metricas']

    cmp_rows.append({
        'Cenário': nome,
        'SP':      SP,
        'Fuzzy TI_f': round(mf['TI_final'], 2),
        'ON/OFF TI_f': round(ro['TI_final'], 2),
        'Fuzzy MAE':  round(mf['MAE'], 3),
        'ON/OFF MAE': round(ro['MAE'], 3),
        'Fuzzy Conv': '✅' if mf['convergiu'] else '❌',
        'ON/OFF Conv': '✅' if ro['convergiu'] else '❌',
        'Fuzzy Overshoot': round(mf['overshoot'], 2),
        'ON/OFF Overshoot': round(ro['overshoot'], 2),
    })

df_cmp = pd.DataFrame(cmp_rows)
print(df_cmp.to_string(index=False))
df_cmp.to_csv('comparacao_fuzzy_onoff.csv', index=False)
print('\n💾 Salvo: comparacao_fuzzy_onoff.csv')

In [ ]:
# ============================================================
# FIGURA 6: FUZZY vs ON/OFF — CURVAS DE TEMPERATURA
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

fig.suptitle('⚖️ GreenHouse: Fuzzy MIMO vs Controle ON/OFF',
             fontsize=13, fontweight='bold', color='#ffcc00')

for ax, (nome, TI, TE, VV, SP, ticks) in zip(axes, CENARIOS):
    rf  = resultados[nome]
    ro  = executar_onoff(TI, TE, VV, SP, n_ticks=ticks)
    
    t_min_f  = [s / 60 for s in rf['hist']['tempo_s']]
    t_min_o  = [s / 60 for s in ro['tempo_s']]
    
    ax.plot(t_min_f, rf['hist']['TI'], color='#00ff88', linewidth=2.5,
            label='Fuzzy')
    ax.plot(t_min_o, ro['TI'],         color='#ff8800', linewidth=2,
            linestyle='--', label='ON/OFF', alpha=0.8)
    ax.axhline(SP,       color='#fff', linewidth=1.5,  linestyle=':', alpha=0.6)
    ax.axhline(SP + 0.5, color='#444', linewidth=0.8,  linestyle=':')
    ax.axhline(SP - 0.5, color='#444', linewidth=0.8,  linestyle=':')
    
    mf_mae = rf['metricas']['MAE']
    oo_mae = ro['MAE']
    ax.set_title(f'{nome}\nFuzzy MAE={mf_mae:.3f}°C | ON/OFF={oo_mae:.3f}°C',
                 fontsize=8, fontweight='bold', color='#e0e0e0')
    ax.set_xlabel('Tempo [min]', fontsize=7)
    ax.set_ylabel('Temp. [°C]', fontsize=7)
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparacao_fuzzy_onoff.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('💾 Salvo: comparacao_fuzzy_onoff.png')

---
## 📄 ETAPA 12 — Relatório Final Consolidado

In [ ]:
# ============================================================
# RELATÓRIO FINAL — RESUMO EXECUTIVO
# ============================================================

print('\n' + '═'*70)
print('  🌿  GreenHouse Fuzzy MIMO Simulator — RELATÓRIO FINAL')
print('  Dissertação de Mestrado — JucemarFC (UFSC)')
print('═'*70)

print('\n📐 ARQUITETURA DO CONTROLADOR')
print(f'   Tipo            : Mamdani MIMO (4 entradas × 2 saídas)')
print(f'   Total de regras : {len(REGRAS)}')
print(f'   Defuzzificação  : Centróide de singletons')
print(f'   Anti-windup     : Zona morta ±0.3°C + ação integral')
print(f'   Ação integral   : Aluminet (bloqueio solar) + boost aquecedor')
print(f'   Tick            : {DT_SIMULACAO}s')

print('\n🌡️  PARÂMETROS FÍSICOS')
print(f'   Estufa          : 8 × 14 × 6 m (672 m³)')
print(f'   Modelo          : Normalizado (7 componentes dT)')
print(f'   TAU_COND        : {TAU_COND}s ({TAU_COND/60:.0f}min)')
print(f'   TAU_EVAP_B      : {TAU_EVAP_B}s')
print(f'   TAU_SUPER_B     : {TAU_SUPER_B}s')
print(f'   HR              : {HR*100:.0f}% | EVAP_EFF : {EVAP_EFF*100:.0f}%')

print('\n📊 RESULTADOS DE VALIDAÇÃO')
n_conv  = sum(1 for n, *_ in CENARIOS if resultados[n]['metricas']['convergiu'])
mae_med = np.mean([resultados[n]['metricas']['MAE'] for n, *_ in CENARIOS])
osh_max = max(resultados[n]['metricas']['overshoot'] for n, *_ in CENARIOS)
print(f'   Convergência    : {n_conv}/{len(CENARIOS)} cenários atingiram SP ±0.5°C')
print(f'   MAE médio       : {mae_med:.4f}°C')
print(f'   Overshoot max   : {osh_max:.2f}°C')

print('\n⚖️  COMPARAÇÃO FUZZY vs ON/OFF')
mae_f = np.mean([resultados[n]['metricas']['MAE'] for n, *_ in CENARIOS])
mae_o_list = []
for nome, TI, TE, VV, SP, ticks in CENARIOS:
    ro = executar_onoff(TI, TE, VV, SP, n_ticks=ticks)
    mae_o_list.append(ro['MAE'])
mae_o = np.mean(mae_o_list)
melhoria = (1 - mae_f / mae_o) * 100 if mae_o > 0 else 0
print(f'   MAE Fuzzy       : {mae_f:.4f}°C')
print(f'   MAE ON/OFF      : {mae_o:.4f}°C')
print(f'   Melhoria        : {melhoria:.1f}%')

print('\n📁 ARQUIVOS GERADOS')
arquivos = [
    'mf_entradas.png', 'resposta_temperatura.png',
    'atuadores_verao.png', 'componentes_termicos.png',
    'ativacao_mf.png', 'metricas_comparativo.png',
    'comparacao_fuzzy_onoff.png', 'comparacao_fuzzy_onoff.png',
    'convergencia_fuzzy.csv', 'comparacao_fuzzy_onoff.csv',
]
for arq in arquivos:
    print(f'   📄 {arq}')

print('\n' + '═'*70)
print('  ✅  SIMULAÇÃO CONCLUÍDA COM SUCESSO!')
print('═'*70)